# Tutorial 5 · Backprop by Hand + micrograd — Part B

**~40 minutes · after the worksheet**

Run reverse-mode differentiation on a graph, then implement the small scalar engine behind it.

Work in pairs. Before each code cell, write down the qualitative result you expect. The notebook is designed to run top-to-bottom in a fresh kernel.


## 1 · A scalar reverse-mode engine


In [1]:
class Value:
    def __init__(self, data, parents=(), backward=lambda: None):
        self.data, self.grad = float(data), 0.0
        self.parents, self._backward = parents, backward
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def back(): self.grad += out.grad; other.grad += out.grad
        out._backward = back; return out
    __radd__ = __add__
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def back(): self.grad += other.data*out.grad; other.grad += self.data*out.grad
        out._backward = back; return out
    __rmul__ = __mul__
    def __pow__(self, power):
        out = Value(self.data**power, (self,))
        def back(): self.grad += power*self.data**(power-1)*out.grad
        out._backward = back; return out
    def backward(self):
        topo, seen = [], set()
        def visit(v):
            if id(v) not in seen:
                seen.add(id(v))
                for parent in v.parents: visit(parent)
                topo.append(v)
        visit(self); self.grad = 1.0
        for v in reversed(topo): v._backward()


## 2 · Reproduce the hand calculation


In [2]:
x, y = Value(2), Value(-3)
a = x*y
b = a+x
loss = b**2
loss.backward()
print("values:", a.data, b.data, loss.data)
print("gradients dx, dy:", x.grad, y.grad)


values: -6.0 -4.0 16.0
gradients dx, dy: 16.0 -16.0


## 3 · Compare with finite differences


In [3]:
def objective(x, y): return (x*y + x)**2
h = 1e-6
dx = (objective(2+h, -3)-objective(2-h, -3))/(2*h)
dy = (objective(2, -3+h)-objective(2, -3-h))/(2*h)
print(dx, dy)


15.999999998683734 -16.000000002236447


## Closing check

Write three sentences: one numerical result you verified, one geometric/probabilistic interpretation, and one failure mode you would now test in a larger implementation.
